# Phase 14 — Le cerveau emprunté, et sa facture

## Objectifs

- Emprunter un modèle de langue déjà entraîné, récupérable librement, assez petit pour tourner sur
  CPU (`distilbert-base-uncased`, 66,4 M de paramètres).
- Le faire travailler sur la tâche des actes précédents (texte expurgé du vocabulaire des formes,
  phase 8), selon trois régimes : poids gelés + petite tête entraînée par-dessus ; une partie du
  modèle autorisée à changer, avec des vitesses différentes selon la profondeur ; de très petites
  valeurs ajoutées (LoRA) à côté d'un modèle resté intact.
- Pour chaque régime : ce que ça donne (score) et ce que ça coûte (paramètres modifiés, temps,
  mémoire, poids à sauvegarder).


## Note de budget de calcul

Un chronométrage préalable de `distilbert-base-uncased` sur cette machine (CPU, lot de 16, longueur 32)
donne environ 1,3 s par pas pour un passage avant + arrière complet, contre 0,2 s pour un passage
avant seul (encodeur gelé). Sur les 58 541 exemples d'entraînement complets, un seul passage
complet à travers les 58 541 relevés couterait déjà plus de deux heures pour le régime le plus lourd.

**Décision, écrite avant toute mesure** : les trois régimes de cette phase tournent sur un
sous-échantillon stratifié fixe de 1 500 relevés d'entraînement et 500 de validation, tiré depuis la
même découpe et le même texte expurgé que la phase 8 — identique pour les trois régimes. Le score de
la phase 8 sur le jeu complet (37,07 %) reste la ligne de référence officielle ; un second repère,
`EmbeddingBag` de la phase 3 réentraîné sur ce même sous-échantillon, est ajouté pour une comparaison
à données strictement égales.


## 1. Imports

In [1]:
from pathlib import Path
import csv
import math
import random
import re
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
import torch
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch import nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer


## 2. Configuration et reproductibilité

In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cpu")
NOM_MODELE_EMPRUNTE = "distilbert-base-uncased"

DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../outputs")
PHASE14_DIR = OUTPUT_DIR / "phase_14_cerveau_emprunte"
PHASE14_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH = DATA_DIR / "releves_klaxo3.csv"

COLUMNS = [
    "datetime", "city", "state", "country", "shape",
    "duration_seconds", "duration_hours_min", "comments",
    "date_posted", "latitude", "longitude",
]

TEST_SIZE = 0.20
SEUIL_MIN_CLASSE = 5
TAILLE_SOUS_ECHANTILLON_TRAIN = 1500
TAILLE_SOUS_ECHANTILLON_VAL = 500
MAX_LENGTH = 32
BATCH_SIZE = 16
N_EPOCHS_MAX = 5
PATIENCE = 2
LORA_RANG = 4


## 3. Reproduction du pipeline de la phase 8 (texte expurgé) et sous-échantillon fixe

In [3]:
if not DATA_PATH.exists():
    raise FileNotFoundError("Lancez d'abord une phase précédente pour télécharger le fichier.")

lignes_valides = []
with open(DATA_PATH, "r", encoding="utf-8", errors="replace", newline="") as f:
    reader = csv.reader(f)
    for row in reader:
        if len(row) == len(COLUMNS):
            lignes_valides.append(row)
df = pd.DataFrame(lignes_valides, columns=COLUMNS)

df["comments_clean"] = df["comments"].fillna("").astype(str).str.strip()
df["shape_clean"] = df["shape"].fillna("").astype(str).str.lower().str.strip()
df["shape_model"] = df["shape_clean"].replace({"round": "circle", "changed": "changing"})

masque_forme_manquante = df["shape_clean"].eq("")
masque_fourre_tout = df["shape_model"].isin(["unknown", "other"])
masque_commentaire_vide = df["comments_clean"].eq("")
df_avant = df.loc[~masque_forme_manquante & ~masque_fourre_tout & ~masque_commentaire_vide].copy()
compte_classes = df_avant["shape_model"].value_counts()
classes_conservees = compte_classes.loc[compte_classes >= SEUIL_MIN_CLASSE].index
df_modele = df_avant.loc[df_avant["shape_model"].isin(classes_conservees)].copy()

def pluriel(mot):
    if mot.endswith(("s", "x", "ch", "sh")):
        return mot + "es"
    if mot.endswith("y") and mot[-2] not in "aeiou":
        return mot[:-1] + "ies"
    return mot + "s"

def tokenizer_simple(texte):
    return re.findall(r"[a-z0-9]+", str(texte).lower())

formes_retenues = sorted(df_modele["shape_model"].unique())
doublons_fusionnes = ["round", "changed"]
variantes_ecriture = {"disk": ["disc"]}
mots_interdits = set()
for mot in list(formes_retenues) + doublons_fusionnes:
    mots_interdits.add(mot)
    mots_interdits.add(pluriel(mot))
    for variante in variantes_ecriture.get(mot, []):
        mots_interdits.add(variante)
        mots_interdits.add(pluriel(variante))

def expurger(texte):
    tokens = tokenizer_simple(texte)
    return " ".join(t for t in tokens if t not in mots_interdits)

df_modele["comments_sans_forme"] = df_modele["comments_clean"].apply(expurger)

X_expurge = df_modele["comments_sans_forme"].copy()
y_cible = df_modele["shape_model"].copy()

index_train, index_val = train_test_split(
    df_modele.index, test_size=TEST_SIZE, random_state=SEED, stratify=y_cible,
)
X_train_complet, X_val_complet = X_expurge.loc[index_train], X_expurge.loc[index_val]
y_train_complet, y_val_complet = y_cible.loc[index_train], y_cible.loc[index_val]

label_encoder = LabelEncoder()
label_encoder.fit(y_cible)
NOMBRE_CLASSES = len(label_encoder.classes_)

_, X_train_sous, _, y_train_sous = train_test_split(
    X_train_complet.reset_index(drop=True), y_train_complet.reset_index(drop=True),
    test_size=TAILLE_SOUS_ECHANTILLON_TRAIN, random_state=SEED, stratify=y_train_complet.reset_index(drop=True),
)
_, X_val_sous, _, y_val_sous = train_test_split(
    X_val_complet.reset_index(drop=True), y_val_complet.reset_index(drop=True),
    test_size=TAILLE_SOUS_ECHANTILLON_VAL, random_state=SEED, stratify=y_val_complet.reset_index(drop=True),
)

print(f"Classes : {NOMBRE_CLASSES} | sous-échantillon train : {len(X_train_sous)} | val : {len(X_val_sous)}")


Classes : 20 | sous-échantillon train : 1500 | val : 500


## 4. Repère à données égales : `EmbeddingBag` de la phase 3 sur ce même sous-échantillon

In [4]:
class DatasetTextesSimple(Dataset):
    def __init__(self, textes, labels, vocabulaire):
        self.textes = list(textes)
        self.labels = list(labels)
        self.vocabulaire = vocabulaire
    def __len__(self):
        return len(self.textes)
    def __getitem__(self, index):
        ids = [self.vocabulaire.get(t, self.vocabulaire["<UNK>"]) for t in tokenizer_simple(self.textes[index])]
        if not ids:
            ids = [self.vocabulaire["<UNK>"]]
        return torch.tensor(ids, dtype=torch.long), int(self.labels[index])

def collate_embedding_bag(batch):
    offsets, tokens_concat, labels_batch = [0], [], []
    for tokens, label in batch:
        tokens_concat.extend(tokens.tolist())
        labels_batch.append(label)
        offsets.append(offsets[-1] + len(tokens))
    return (torch.tensor(tokens_concat, dtype=torch.long), torch.tensor(offsets[:-1], dtype=torch.long), torch.tensor(labels_batch, dtype=torch.long))

vocabulaire_repere = {"<PAD>": 0, "<UNK>": 1}
for texte in X_train_sous:
    for token in tokenizer_simple(texte):
        if token not in vocabulaire_repere:
            vocabulaire_repere[token] = len(vocabulaire_repere)

y_train_sous_ids = label_encoder.transform(y_train_sous)
y_val_sous_ids = label_encoder.transform(y_val_sous)

loader_train_repere = DataLoader(DatasetTextesSimple(X_train_sous, y_train_sous_ids, vocabulaire_repere), batch_size=64, shuffle=True, collate_fn=collate_embedding_bag)
loader_val_repere = DataLoader(DatasetTextesSimple(X_val_sous, y_val_sous_ids, vocabulaire_repere), batch_size=64, shuffle=False, collate_fn=collate_embedding_bag)

class ClassifieurEmbeddingBag(nn.Module):
    def __init__(self, taille_vocab, n_classes, dim=96, hidden=128):
        super().__init__()
        self.embedding = nn.EmbeddingBag(taille_vocab, dim, mode="mean")
        self.reseau = nn.Sequential(nn.Linear(dim, hidden), nn.ReLU(), nn.Dropout(0.3), nn.Linear(hidden, n_classes))
    def forward(self, tokens, offsets):
        return self.reseau(self.embedding(tokens, offsets))

torch.manual_seed(SEED)
modele_repere = ClassifieurEmbeddingBag(len(vocabulaire_repere), NOMBRE_CLASSES)
opt_repere = torch.optim.AdamW(modele_repere.parameters(), lr=0.003, weight_decay=1e-4)
perte_repere_fn = nn.CrossEntropyLoss()

meilleure_perte, meilleur_etat, sans_amelio = float("inf"), None, 0
for epoch in range(1, 16):
    modele_repere.train()
    for tokens, offsets, labels in loader_train_repere:
        opt_repere.zero_grad()
        perte = perte_repere_fn(modele_repere(tokens, offsets), labels)
        perte.backward()
        opt_repere.step()
    modele_repere.eval()
    preds, reels, perte_val_tot, n_val = [], [], 0.0, 0
    with torch.no_grad():
        for tokens, offsets, labels in loader_val_repere:
            logits = modele_repere(tokens, offsets)
            perte_val_tot += perte_repere_fn(logits, labels).item() * len(labels)
            n_val += len(labels)
            preds.extend(logits.argmax(dim=1).tolist())
            reels.extend(labels.tolist())
    perte_val = perte_val_tot / n_val
    if perte_val < meilleure_perte:
        meilleure_perte, sans_amelio = perte_val, 0
        meilleur_etat = {k: v.clone() for k, v in modele_repere.state_dict().items()}
    else:
        sans_amelio += 1
    if sans_amelio >= 3:
        break
modele_repere.load_state_dict(meilleur_etat)
modele_repere.eval()
preds = []
with torch.no_grad():
    for tokens, offsets, labels in loader_val_repere:
        preds.extend(modele_repere(tokens, offsets).argmax(dim=1).tolist())
accuracy_repere_embeddingbag = accuracy_score(y_val_sous_ids, preds)
print(f"Repère EmbeddingBag (même sous-échantillon) : {accuracy_repere_embeddingbag:.2%}")


Repère EmbeddingBag (même sous-échantillon) : 30.40%


## 5. Le modèle emprunté : tokenizer, jeu de données, pooling `[CLS]`

In [5]:
tokenizer_emprunte = AutoTokenizer.from_pretrained(NOM_MODELE_EMPRUNTE)

class DatasetTransformer(Dataset):
    def __init__(self, textes, labels, tokenizer, max_length):
        self.encodages = tokenizer(
            list(textes), truncation=True, padding="max_length", max_length=max_length, return_tensors="pt",
        )
        self.labels = torch.tensor(list(labels), dtype=torch.long)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, index):
        return {
            "input_ids": self.encodages["input_ids"][index],
            "attention_mask": self.encodages["attention_mask"][index],
            "labels": self.labels[index],
        }

dataset_train_transformer = DatasetTransformer(X_train_sous, y_train_sous_ids, tokenizer_emprunte, MAX_LENGTH)
dataset_val_transformer = DatasetTransformer(X_val_sous, y_val_sous_ids, tokenizer_emprunte, MAX_LENGTH)

loader_train_transformer = DataLoader(dataset_train_transformer, batch_size=BATCH_SIZE, shuffle=True)
loader_val_transformer = DataLoader(dataset_val_transformer, batch_size=BATCH_SIZE, shuffle=False)

class ClassifieurTransformer(nn.Module):
    def __init__(self, encodeur, dimension_cachee, n_classes):
        super().__init__()
        self.encodeur = encodeur
        self.tete = nn.Sequential(nn.Dropout(0.1), nn.Linear(dimension_cachee, n_classes))
    def forward(self, input_ids, attention_mask):
        sortie = self.encodeur(input_ids=input_ids, attention_mask=attention_mask)
        cls = sortie.last_hidden_state[:, 0]  # jeton [CLS]
        return self.tete(cls)


## 6. Outil de mesure commun (temps, mémoire, paramètres modifiés, poids à sauvegarder)

In [6]:
def compter_parametres_entrainables(modele):
    return sum(p.numel() for p in modele.parameters() if p.requires_grad)

def taille_poids_entrainables_mo(modele):
    return sum(p.numel() * 4 for p in modele.parameters() if p.requires_grad) / (1024 ** 2)

def entrainer_et_mesurer_transformer(nom, modele, groupes_parametres, n_epochs=N_EPOCHS_MAX, patience=PATIENCE):
    optimiseur = torch.optim.AdamW(groupes_parametres)
    fonction_perte = nn.CrossEntropyLoss()

    processus = psutil.Process()
    memoire_max_mo = processus.memory_info().rss / (1024 ** 2)

    historique = {"epoch": [], "train_loss": [], "val_loss": [], "val_acc": []}
    meilleure_perte_val, meilleur_etat, sans_amelio = float("inf"), None, 0

    debut = time.perf_counter()
    temps_premier_pas = None
    for epoch in range(1, n_epochs + 1):
        modele.train()
        perte_totale, n_ex = 0.0, 0
        for i, batch in enumerate(loader_train_transformer):
            debut_pas = time.perf_counter()
            optimiseur.zero_grad()
            logits = modele(batch["input_ids"], batch["attention_mask"])
            perte = fonction_perte(logits, batch["labels"])
            perte.backward()
            optimiseur.step()
            if temps_premier_pas is None and epoch == 1 and i == 2:
                temps_premier_pas = time.perf_counter() - debut_pas
            perte_totale += perte.item() * len(batch["labels"])
            n_ex += len(batch["labels"])
            memoire_max_mo = max(memoire_max_mo, processus.memory_info().rss / (1024 ** 2))
        perte_train = perte_totale / n_ex

        modele.eval()
        perte_val_tot, n_val, preds, reels = 0.0, 0, [], []
        with torch.no_grad():
            for batch in loader_val_transformer:
                logits = modele(batch["input_ids"], batch["attention_mask"])
                perte_val_tot += fonction_perte(logits, batch["labels"]).item() * len(batch["labels"])
                n_val += len(batch["labels"])
                preds.extend(logits.argmax(dim=1).tolist())
                reels.extend(batch["labels"].tolist())
        perte_val = perte_val_tot / n_val
        acc_val = accuracy_score(reels, preds)

        historique["epoch"].append(epoch)
        historique["train_loss"].append(perte_train)
        historique["val_loss"].append(perte_val)
        historique["val_acc"].append(acc_val)
        print(f"[{nom}] epoch {epoch} | train={perte_train:.4f} | val={perte_val:.4f} | acc={acc_val:.2%}")

        if perte_val < meilleure_perte_val:
            meilleure_perte_val, sans_amelio = perte_val, 0
            meilleur_etat = {k: v.clone() for k, v in modele.state_dict().items()}
        else:
            sans_amelio += 1
        if sans_amelio >= patience:
            print(f"[{nom}] arrêt anticipé à l'époque {epoch}.")
            break

    temps_total = time.perf_counter() - debut
    modele.load_state_dict(meilleur_etat)

    return {
        "nom": nom, "historique": historique, "temps_total": temps_total,
        "temps_par_pas_s": temps_premier_pas, "accuracy_finale": max(historique["val_acc"]),
        "memoire_max_mo": memoire_max_mo,
        "parametres_entrainables": compter_parametres_entrainables(modele),
        "poids_a_sauvegarder_mo": taille_poids_entrainables_mo(modele),
    }


## 7. Régime 1 — poids gelés, petite tête entraînée par-dessus

Aucune valeur interne de l'encodeur ne change. Ce qu'on entraîne est minuscule : la tête de
classification posée par-dessus.

In [7]:
encodeur_1 = AutoModel.from_pretrained(NOM_MODELE_EMPRUNTE)
for p in encodeur_1.parameters():
    p.requires_grad = False

modele_regime1 = ClassifieurTransformer(encodeur_1, encodeur_1.config.dim, NOMBRE_CLASSES)
groupes_regime1 = [{"params": modele_regime1.tete.parameters(), "lr": 1e-3}]

resultat_regime1 = entrainer_et_mesurer_transformer("regime_1_geles", modele_regime1, groupes_regime1)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[regime_1_geles] epoch 1 | train=2.5190 | val=2.4641 | acc=25.20%


[regime_1_geles] epoch 2 | train=2.4091 | val=2.4134 | acc=25.60%


[regime_1_geles] epoch 3 | train=2.3542 | val=2.4103 | acc=25.20%


[regime_1_geles] epoch 4 | train=2.3228 | val=2.4056 | acc=25.20%


[regime_1_geles] epoch 5 | train=2.2666 | val=2.3925 | acc=29.40%


## 8. Régime 2 — une partie du modèle autorisée à changer, à des vitesses différentes

Les 4 premières couches du transformeur (les plus proches de l'entrée) et les embeddings restent
gelés. Les 2 dernières couches (les plus proches de la sortie) sont dégelées, avec un taux
d'apprentissage plus prudent que la tête de classification — ce qui est loin de la sortie a moins de
raisons de bouger vite que ce qui en est proche.

In [8]:
encodeur_2 = AutoModel.from_pretrained(NOM_MODELE_EMPRUNTE)
for p in encodeur_2.parameters():
    p.requires_grad = False

couches_transformeur = encodeur_2.transformer.layer
NOMBRE_COUCHES_DEGELEES = 2
for couche in couches_transformeur[-NOMBRE_COUCHES_DEGELEES:]:
    for p in couche.parameters():
        p.requires_grad = True

modele_regime2 = ClassifieurTransformer(encodeur_2, encodeur_2.config.dim, NOMBRE_CLASSES)

groupes_regime2 = [
    {"params": couches_transformeur[-2].parameters(), "lr": 5e-6},   # plus loin de la sortie : plus prudent
    {"params": couches_transformeur[-1].parameters(), "lr": 2e-5},   # plus proche de la sortie
    {"params": modele_regime2.tete.parameters(), "lr": 1e-3},        # la tête, jamais gelée, la plus rapide
]

resultat_regime2 = entrainer_et_mesurer_transformer("regime_2_partiel", modele_regime2, groupes_regime2)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[regime_2_partiel] epoch 1 | train=2.5158 | val=2.4337 | acc=25.00%


[regime_2_partiel] epoch 2 | train=2.3322 | val=2.3496 | acc=30.60%


[regime_2_partiel] epoch 3 | train=2.1714 | val=2.3143 | acc=31.00%


[regime_2_partiel] epoch 4 | train=2.0011 | val=2.3124 | acc=32.00%


[regime_2_partiel] epoch 5 | train=1.7918 | val=2.4039 | acc=31.20%


## 9. Régime 3 — LoRA : le modèle emprunté reste intact, on n'entraîne que de petits ajouts

Une couche `nn.Linear` gelée reçoit un chemin parallèle de rang très bas (`A` puis `B`, initialisé à
zéro pour ne rien changer au démarrage) : `sortie = Wx + (B(A(x))) × échelle`. Seuls `A` et `B` sont
entraînés, appliqués aux projections « question » et « contenu » de l'attention de chaque couche —
exactement les rôles définis au tableau de la phase 10.

In [9]:
class LoRALinear(nn.Module):
    def __init__(self, couche_lineaire_gelee, rang=LORA_RANG, echelle=1.0):
        super().__init__()
        self.lineaire_gele = couche_lineaire_gelee
        for p in self.lineaire_gele.parameters():
            p.requires_grad = False
        dim_in, dim_out = couche_lineaire_gelee.in_features, couche_lineaire_gelee.out_features
        self.A = nn.Parameter(torch.randn(dim_in, rang) * 0.01)
        self.B = nn.Parameter(torch.zeros(rang, dim_out))  # zero : le modele emprunte n'est pas modifie au depart
        self.echelle = echelle

    def forward(self, x):
        return self.lineaire_gele(x) + (x @ self.A @ self.B) * self.echelle

encodeur_3 = AutoModel.from_pretrained(NOM_MODELE_EMPRUNTE)
for p in encodeur_3.parameters():
    p.requires_grad = False

for couche in encodeur_3.transformer.layer:
    attention = couche.attention
    attention.q_lin = LoRALinear(attention.q_lin)
    attention.v_lin = LoRALinear(attention.v_lin)

modele_regime3 = ClassifieurTransformer(encodeur_3, encodeur_3.config.dim, NOMBRE_CLASSES)

parametres_lora = [p for n, p in modele_regime3.named_parameters() if n.endswith(".A") or n.endswith(".B")]
groupes_regime3 = [
    {"params": parametres_lora, "lr": 1e-3},
    {"params": modele_regime3.tete.parameters(), "lr": 1e-3},
]

resultat_regime3 = entrainer_et_mesurer_transformer("regime_3_lora", modele_regime3, groupes_regime3)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[regime_3_lora] epoch 1 | train=2.4983 | val=2.3688 | acc=27.20%


[regime_3_lora] epoch 2 | train=2.2727 | val=2.2996 | acc=31.00%


[regime_3_lora] epoch 3 | train=2.0702 | val=2.2750 | acc=30.00%


[regime_3_lora] epoch 4 | train=1.8704 | val=2.2985 | acc=31.20%


[regime_3_lora] epoch 5 | train=1.6430 | val=2.3418 | acc=31.40%
[regime_3_lora] arrêt anticipé à l'époque 5.


## 10. Tableau comparatif : ce que chaque régime donne, ce qu'il coûte

In [10]:
def taille_totale_mo(modele):
    return sum(p.numel() * 4 for p in modele.parameters()) / (1024 ** 2)

tableau_phase14 = pd.DataFrame([
    {
        "regime": r["nom"],
        "accuracy": r["accuracy_finale"],
        "parametres_entrainables": r["parametres_entrainables"],
        "part_des_66M_parametres": r["parametres_entrainables"] / 66_362_880,
        "temps_total_s": r["temps_total"],
        "temps_par_pas_ms": (r["temps_par_pas_s"] or float("nan")) * 1000,
        "memoire_max_mo": r["memoire_max_mo"],
        "poids_a_sauvegarder_mo": r["poids_a_sauvegarder_mo"],
    }
    for r in [resultat_regime1, resultat_regime2, resultat_regime3]
])

ligne_reference = pd.DataFrame([{
    "regime": "reference_phase8_jeu_complet", "accuracy": 0.3707,
    "parametres_entrainables": None, "part_des_66M_parametres": None,
    "temps_total_s": None, "temps_par_pas_ms": None, "memoire_max_mo": None, "poids_a_sauvegarder_mo": None,
}])
ligne_repere = pd.DataFrame([{
    "regime": "repere_embeddingbag_meme_sous_echantillon", "accuracy": accuracy_repere_embeddingbag,
    "parametres_entrainables": None, "part_des_66M_parametres": None,
    "temps_total_s": None, "temps_par_pas_ms": None, "memoire_max_mo": None, "poids_a_sauvegarder_mo": None,
}])

tableau_complet = pd.concat([ligne_reference, ligne_repere, tableau_phase14], ignore_index=True)
tableau_complet


,regime,accuracy,parametres_entrainables,part_des_66M_parametres,temps_total_s,temps_par_pas_ms,memoire_max_mo,poids_a_sauvegarder_mo
0,reference_phase8_jeu_complet,0.3707,None,None,None,None,None,None
1,repere_embeddingbag_meme_sous_echantillon,0.3040,None,None,None,None,None,None
2,regime_1_geles,0.2940,15380,0.000232,469.661796,815.074,1458.21875,0.05867
3,regime_2_partiel,0.3200,14191124,0.213841,717.875064,1299.6728,1925.941406,54.134842
4,regime_3_lora,0.3140,89108,0.001343,796.358701,1490.9466,1861.878906,0.33992


## 11. Tranche : lequel le Bureau peut se payer

In [11]:
meilleur_regime = tableau_phase14.loc[tableau_phase14["accuracy"].idxmax()]

print(f"Meilleur régime sur ce sous-échantillon : {meilleur_regime['regime']} ({meilleur_regime['accuracy']:.2%})")
print(f"Repère EmbeddingBag, même sous-échantillon : {accuracy_repere_embeddingbag:.2%}")
print()
for _, ligne in tableau_phase14.iterrows():
    print(
        f"{ligne['regime']:20s} | accuracy {ligne['accuracy']:.2%} | "
        f"{ligne['parametres_entrainables']:>9,} paramètres entraînables "
        f"({ligne['part_des_66M_parametres']:.2%} du modèle) | "
        f"{ligne['poids_a_sauvegarder_mo']:.2f} Mo à sauvegarder | "
        f"{ligne['temps_total_s']:.1f} s au total"
    )


Meilleur régime sur ce sous-échantillon : regime_2_partiel (32.00%)
Repère EmbeddingBag, même sous-échantillon : 30.40%

regime_1_geles       | accuracy 29.40% |    15,380 paramètres entraînables (0.02% du modèle) | 0.06 Mo à sauvegarder | 469.7 s au total
regime_2_partiel     | accuracy 32.00% | 14,191,124 paramètres entraînables (21.38% du modèle) | 54.13 Mo à sauvegarder | 717.9 s au total
regime_3_lora        | accuracy 31.40% |    89,108 paramètres entraînables (0.13% du modèle) | 0.34 Mo à sauvegarder | 796.4 s au total


**Verdict, écrit après lecture du tableau ci-dessus** — à compléter dans `RAPPORTS.md` une fois
les chiffres réels connus : le régime retenu est celui qui offre le meilleur compromis entre le score
obtenu, la part du modèle réellement modifiée et le poids du fichier à livrer, pas nécessairement celui
qui obtient l'accuracy la plus haute dans l'absolu.

## 12. Export

In [12]:
tableau_complet.to_csv(PHASE14_DIR / "tableau_comparatif.csv", index=False)
for r in [resultat_regime1, resultat_regime2, resultat_regime3]:
    pd.DataFrame(r["historique"]).to_csv(PHASE14_DIR / f"historique_{r['nom']}.csv", index=False)

resume_phase14 = pd.DataFrame([{
    "modele_emprunte": NOM_MODELE_EMPRUNTE,
    "parametres_modele_emprunte": 66_362_880,
    "taille_sous_echantillon_train": TAILLE_SOUS_ECHANTILLON_TRAIN,
    "accuracy_reference_phase8_jeu_complet": 0.3707,
    "accuracy_repere_embeddingbag_sous_echantillon": accuracy_repere_embeddingbag,
    "accuracy_regime1_geles": resultat_regime1["accuracy_finale"],
    "accuracy_regime2_partiel": resultat_regime2["accuracy_finale"],
    "accuracy_regime3_lora": resultat_regime3["accuracy_finale"],
}])
resume_phase14.to_csv(PHASE14_DIR / "resume_phase14.csv", index=False)
resume_phase14


,modele_emprunte,parametres_modele_emprunte,taille_sous_echantillon_train,accuracy_reference_phase8_jeu_complet,accuracy_repere_embeddingbag_sous_echantillon,accuracy_regime1_geles,accuracy_regime2_partiel,accuracy_regime3_lora
0,distilbert-base-uncased,66362880,1500,0.3707,0.304,0.294,0.32,0.314
